In [49]:
from transformers import pipeline

model = "Qwen/Qwen2.5-1.5B"

generator = pipeline(
    "text-generation",
     model=model,
     max_length=None,
     max_new_tokens=200, 
     do_sample=True,
     return_full_text=False
)

prompt = "I'm studying data science at UBC in Vancouver because I want to work as a"

outputs = generator(prompt, max_new_tokens=30)

print(outputs[0]["generated_text"])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

 data scientist in Canada. I was previously an undergraduate student in Computer Science majoring in Algorithms and Theory. I have also done some research in the field


In [33]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from app.app import load_resources

In [34]:
documents, bm25 = load_resources()

In [7]:
query = "something to keep your face moisturized all day"

semantic_results = semantic_search(
        documents,
        query,
        k=5,
        sample_size=10000,
        reload_index=False
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
for doc, score in semantic_results:
    print("Score: ", score)
    print("Product: ", doc.metadata.get("product_title"))
    print("Review: ", doc.metadata.get("product_review"))
    print()


Score:  0.65787405
Product:  Avon Care Rich Moisture Comforting Nourishing Cream with soybean 6.7 Fl Oz
Review:  the face cream is good for me

Score:  0.6580241
Product:  2pcs Green Tea Purifying Clay Stick Mask, Face Moisturizes Oil Control, Deep Clean Pore, Improves Skin,for All Skin Types Men Women
Review:  Over dried my face.

Score:  0.6597388
Product:  DayTime Moisturizer for Dry Skin
Review:  work good

Score:  0.6666714
Product:  Hydrogel Face Masks (5 MASKS) with Aloe, Matcha Green Tea, & Caffeine; WHOLESALE Gel Sheet Masks for Anti-Aging, Deep Hydration - Collagen and Antioxidants Repair Sun Damage, Brighten Skin
Review:  Makes my skin feel great.

Score:  0.6695756
Product:  Paula's Choice Rehydrating Moisture Mask with Plant Oils Antioxidants and Amino Acids, 3 Ounce Bottle, Anti-Aging Face Mask for Replenishing Dry or Very Dry Skin Vitamin E Face Mask with Jojoba
Review:  Best moisturizer I have found



In [7]:
from src.semantic import create_faiss_index

vectorstore = create_faiss_index(documents, 10000, reload_index=False)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=generator)

In [51]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
You must answer using ONLY the information in the context.

- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.

Context:
{context}

Question:
{input}

Answer:
"""
)

In [62]:
SYSTEM_PROMPT = """
    You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible."""

def build_prompt(query, context):
    return f"""{SYSTEM_PROMPT}

context:
{context}

question: 
{query}

Recommend ONE product using the context.
Do not add additional explanations or repeat the prompt.
Stop after the recommendation.

Return the answer exactly in this format:

Product Title:
Product ASIN:
Reason for Recommendation: Write 2 concise sentences explaining why the product matches what the user is looking for.

Stop after the reason.
"""

In [63]:
def build_context(docs):
    return "\n\n".join(
        f"Product ASIN: {doc.metadata.get('asin')}\n"
        f"Product Title: {doc.metadata.get('product_title')}\n"
        f"Product Rating: {doc.metadata.get('product_rating')}\n"
        f"Product Review: {doc.metadata.get('product_review')}\n"
        for doc in docs
    )

In [64]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

format_context = RunnableLambda(build_context)
def prompt_builder(inputs):
    return build_prompt(inputs["input"], inputs["context"])

prompt = RunnableLambda(prompt_builder)


rag_chain = (
    {
        "context": retriever | format_context,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [65]:
query = "Moisturizing shampoo for thick curly hair"

response = rag_chain.invoke(query)

print(response)

Product Title: THE MANE CHOICE- Easy On The Curls Detangling & Hydration Conditioner - Biotin, Avocado Oil and Vitam E to Clean, Nourish & Hydrate Your Curly Hair (8 Ounces / 230 Milliliters)
Product ASIN: B01ETV11XU
Reason for Recommendation:
Moisturizing shampoo for thick curly hair: The THE MANE CHOICE- Easy On The Curls Detangling & Hydration Conditioner - Biotin, Avocado Oil and Vitam E to Clean, Nourish & Hydrate Your Curly Hair (8 Ounces / 230 Milliliters) is perfect for moisturizing your thick curly hair. It contains biotin, avocado oil, and vitamins E to provide much-needed hydration and nourishment to your hair. Its easy to use formula makes it a great choice for those with dry or wavy curly
